# CORRECTION DRONE — Hardware Design on Kaggle GPU
## Target: Complete BOM, sensor simulation, power budget, latency model

DESIGN REQUIREMENTS:
  • Real-time joint angle correction (<50ms latency)
  • 6 joints × 3-axis IMU = 18 sensor channels
  • Drone-to-suit wireless link with <10ms latency
  • Edge AI processor for on-board correction prediction
  • 40+ minute battery life (matches drone hover time)
  • Total BOM cost target: <$5,000 (consumer add-on to existing suit)

OUTPUTS:
  • Hardware BOM with costs
  • Sensor fusion simulation
  • Power budget calculation
  • Latency budget analysis
  • 3D sensor placement optimization

In [ ]:
# HARDWARE BILL OF MATERIALS
import json, math, numpy as np

BOM = {
    'drone_hardware': [
        {'part': 'Raspberry Pi 5 (8GB)', 'qty': 1, 'cost': 80, 'purpose': 'Edge AI processor on drone'},
        {'part': 'Intel RealSense D455', 'qty': 1, 'cost': 350, 'purpose': 'Depth camera for 3D pose estimation'},
        {'part': 'Raspberry Pi Camera Module 3', 'qty': 1, 'cost': 25, 'purpose': 'Secondary RGB camera'},
        {'part': 'ESP32-S3', 'qty': 1, 'cost': 5, 'purpose': 'Low-latency radio link (ESP-NOW)'},
        {'part': 'MPU-6050 IMU (6-axis)', 'qty': 1, 'cost': 2, 'purpose': 'Drone stabilization reference'},
        {'part': 'Lipo 4S 5000mAh', 'qty': 2, 'cost': 40, 'purpose': 'Drone + compute power (total 10Ah)'},
        {'part': 'Drone frame + motors + props', 'qty': 1, 'cost': 150, 'purpose': 'DJI F450 clone quadcopter kit'},
        {'part': '3-axis brushless gimbal', 'qty': 1, 'cost': 80, 'purpose': 'Camera stabilization'},
    ],
    'suit_hardware': [
        {'part': 'MPU-9250 IMU (9-axis)', 'qty': 6, 'cost': 30, 'purpose': 'One per joint (toe,ankle,knee,hip,shoulder,neck)'},
        {'part': 'ESP32-C3', 'qty': 6, 'cost': 18, 'purpose': 'One per joint — reads IMU, sends via ESP-NOW'},
        {'part': 'Strain gauge + HX711', 'qty': 4, 'cost': 16, 'purpose': 'Load sensing on knee + hip joints'},
        {'part': 'Haptic motor (vibration)', 'qty': 6, 'cost': 12, 'purpose': 'Per-joint correction feedback to pilot'},
        {'part': 'Lipo 3S 3000mAh', 'qty': 1, 'cost': 25, 'purpose': 'Suit sensor power'},
        {'part': 'ESP32-S3 (main hub)', 'qty': 1, 'cost': 5, 'purpose': 'Aggregates joint data, sends to drone'},
    ],
    'software': [
        {'part': 'MediaPipe Pose (on-device)', 'qty': 1, 'cost': 0, 'purpose': '2D pose estimation fallback'},
        {'part': 'Custom Pytorch model (on RPi5)', 'qty': 1, 'cost': 0, 'purpose': 'Trained correction predictor'},
        {'part': 'ESP-NOW firmware', 'qty': 7, 'cost': 0, 'purpose': 'Wireless mesh between joints+drone'},
    ]}

# Calculate totals
drone_cost = sum(item['cost']*item['qty'] for item in BOM['drone_hardware'])
suit_cost = sum(item['cost']*item['qty'] for item in BOM['suit_hardware'])
sw_cost = sum(item['cost']*item['qty'] for item in BOM['software'])
total = drone_cost + suit_cost + sw_cost

print('='*60)
print('CORRECTION DRONE — Hardware BOM')
print('='*60)
print(f'  Drone hardware:  ${drone_cost}')
for item in BOM['drone_hardware']:
    print(f'    {item["qty"]}x {item["part"]:<30s} ${item["cost"]*item["qty"]:>5d}  ({item["purpose"]})')
print(f'  Suit hardware:   ${suit_cost}')
for item in BOM['suit_hardware']:
    print(f'    {item["qty"]}x {item["part"]:<30s} ${item["cost"]*item["qty"]:>5d}  ({item["purpose"]})')
print(f'  Software:        ${sw_cost} (open source)')
print(f'  TOTAL BOM:       ${total}')
print(f'  Target:          <$5,000')
print(f'  Status:          {"✓ UNDER BUDGET" if total < 5000 else "✗ OVER BUDGET"}')

In [ ]:
# SENSOR FUSION SIMULATION
# Simulate 6 IMUs reading joint angles with noise
# Compare against ideal checkpoint angles

JOINTS = {
    'toe':{'min':-30,'max':45,'phase':0},'ankle':{'min':-15,'max':25,'phase':5},
    'knee':{'min':-30,'max':5,'phase':15},'hip':{'min':-20,'max':15,'phase':30},
    'shoulder':{'min':-12,'max':12,'phase':180},'neck':{'min':-3,'max':3,'phase':90}}

def ideal_angle(joint, frame, total=6):
    j = JOINTS[joint]
    t = frame/total*2*math.pi
    return j['min']+(j['max']-j['min'])*(1+math.sin(t+math.radians(j['phase'])))/2

def simulate_imu(joint, frame, noise_density=0.005, drift=0.1):
    """Simulate MPU-9250 reading with realistic noise."""
    ideal = ideal_angle(joint, frame)
    gaussian_noise = np.random.normal(0, noise_density * abs(ideal))
    bias_drift = np.random.normal(0, drift / 3600)  # deg/s drift
    return ideal + gaussian_noise + bias_drift

def simulate_depth_camera(joint, frame, error_mm=5, distance_m=3):
    """Intel RealSense depth error model."""
    ideal = ideal_angle(joint, frame)
    # Angular error from depth error: error = atan(error_mm / (distance_m * 1000))
    angular_error = math.degrees(math.atan(error_mm / (distance_m * 1000)))
    return ideal + np.random.normal(0, angular_error / 3)  # 3-sigma

# Simulate one full cycle
print('='*60)
print('SENSOR FUSION — One Movement Cycle')
print('='*60)
print(f'{"Joint":8s} {"Ideal":>6s} {"IMU":>6s} {"Camera":>6s} {"Fused":>6s} {"Error":>6s}')
total_error_imu = 0
total_error_cam = 0
for joint in JOINTS:
    for frame in range(6):
        ideal = ideal_angle(joint, frame)
        imu = simulate_imu(joint, frame)
        cam = simulate_depth_camera(joint, frame)
        # Simple fusion: average with confidence weighting
        # IMU is noisier but faster. Camera is slower but more accurate at short range.
        fused = imu * 0.4 + cam * 0.6  # Weight toward camera at 3m
        error = abs(fused - ideal)
        total_error_imu += abs(imu-ideal)
        total_error_cam += abs(cam-ideal)
        if frame == 3:  # Print mid-cycle
            print(f'  {joint:8s} {ideal:6.1f} {imu:6.1f} {cam:6.1f} {fused:6.1f} {error:6.2f}')

print(f'  Mean IMU error: {total_error_imu/36:.3f} deg')
print(f'  Mean Cam error: {total_error_cam/36:.3f} deg')
print(f'  IMU+Camera fusion reduces error by ~{(1-(min(total_error_imu,total_error_cam)/max(total_error_imu,total_error_cam)))*100:.0f}%')

In [ ]:
# POWER BUDGET
print('='*60)
print('POWER BUDGET')
print('='*60)

power = {
    'drone': [
        ('Motors (hover)', 120, '4x 2212 brushless'),
        ('Raspberry Pi 5', 15, 'Under load'),
        ('RealSense D455', 4, 'Streaming'),
        ('ESP32-S3', 0.5, 'ESP-NOW transmit'),
        ('Camera gimbal', 5, '3-axis stabilization'),
        ('Total drone', 144.5, ''),
    ],
    'suit': [
        ('6x MPU-9250', 0.03, '5mA each'),
        ('6x ESP32-C3', 1.2, '200mA each, deep sleep between frames'),
        ('6x Haptic motors', 1.8, '300mA each, pulsed'),
        ('Main ESP32 hub', 0.4, 'Aggregating + forwarding'),
        ('Total suit', 3.43, ''),
    ]}

drone_total = 0
for name, watts, note in power['drone']:
    print(f'  {name:<25s} {watts:6.1f}W  ({note})')
    if 'Total' not in name:
        drone_total += watts

print()
suit_total = 0
for name, watts, note in power['suit']:
    print(f'  {name:<25s} {watts:6.1f}W  ({note})')
    if 'Total' not in name:
        suit_total += watts

# Battery life
drone_battery_wh = 14.8 * 5  # 4S 5000mAh = 74Wh
drone_life_min = drone_battery_wh / drone_total * 60

suit_battery_wh = 11.1 * 3  # 3S 3000mAh = 33.3Wh
suit_life_h = suit_battery_wh / suit_total

print(f'\n  Drone consumption: {drone_total:.1f}W')
print(f'  Drone battery: 74Wh (4S 5000mAh)')
print(f'  Drone flight time: {drone_life_min:.0f} min')
print(f'  Suit consumption: {suit_total:.1f}W')
print(f'  Suit battery: 33.3Wh (3S 3000mAh)')
print(f'  Suit runtime: {suit_life_h:.1f} hours')

In [ ]:
# LATENCY BUDGET — Must be <50ms total for real-time correction
print('='*60)
print('LATENCY BUDGET (Target: <50ms)')
print('='*60)

latency = [
    ('IMU read (MPU-9250)', 2, 'I2C at 400kHz'),
    ('ESP32 processing', 1, 'Onboard filtering'),
    ('ESP-NOW transmit', 3, '2.4GHz, <250 bytes'),
    ('ESP32 hub aggregation', 2, 'Collect 6 joint packets'),
    ('ESP-NOW to drone', 3, 'Suit → Drone uplink'),
    ('Deep camera capture', 10, 'RealSense D455 depth frame'),
    ('RPi5 sensor fusion', 5, 'IMU + camera Kalman filter'),
    ('Correction model inference', 8, 'Trained Ridge on RPi5'),
    ('ESP-NOW correction downlink', 3, 'Drone → Suit per-joint delta'),
    ('Haptic motor activation', 2, 'PWM to vibration motor'),
    ('Human reaction time', 0, 'Not included — pilot is the actuator'),
]

total = 0
for name, ms, note in latency:
    total += ms
    bar = chr(9608) * min(20, ms) + chr(9617) * max(0, 20-ms)
    print(f'  {name:<35s} {ms:2d}ms {bar}')

print(f'  {"TOTAL SYSTEM LATENCY":35s} {total}ms')
print(f'  Status: {"✓ UNDER 50ms" if total < 50 else "✗ OVER — optimize"}')
print(f'  Margin: {50-total}ms remaining')

In [ ]:
# SENSOR PLACEMENT OPTIMIZATION
# Where to mount each sensor for best joint angle measurement

placement = {
    'toe':      {'location': 'Metatarsal head', 'axis': 'Pitch', 'mount': 'Shoe clip', 'occlusion_risk': 'Low'},
    'ankle':    {'location': 'Lateral malleolus', 'axis': 'Pitch', 'mount': 'Ankle strap', 'occlusion_risk': 'Low'},
    'knee':     {'location': 'Lateral epicondyle', 'axis': 'Pitch', 'mount': 'Knee brace', 'occlusion_risk': 'Medium (suit plating)'},
    'hip':      {'location': 'Greater trochanter', 'axis': 'Pitch+Roll', 'mount': 'Waist belt', 'occlusion_risk': 'Medium'},
    'shoulder': {'location': 'Acromion process', 'axis': 'Pitch+Roll+Yaw', 'mount': 'Shoulder strap', 'occlusion_risk': 'High (arm movement)'},
    'neck':     {'location': 'C7 vertebra', 'axis': 'Pitch+Roll', 'mount': 'Collar clip', 'occlusion_risk': 'Low'},
}

print('='*60)
print('SENSOR PLACEMENT — Optimized for Joint Angle Measurement')
print('='*60)
for joint, spec in placement.items():
    print(f'  {joint:8s}: {spec["location"]:<25s} | {spec["axis"]:<12s} | {spec["mount"]:<15s} | occ={spec["occlusion_risk"]}')

In [ ]:
# EXPORT HARDWARE SPEC

hardware_spec = {
    'bom': BOM,
    'total_cost': drone_cost + suit_cost,
    'power': {
        'drone_watts': drone_total,
        'suit_watts': suit_total,
        'drone_battery_life_min': round(drone_life_min, 1),
        'suit_battery_life_hours': round(suit_life_h, 1)},
    'latency': {
        'total_ms': total,
        'under_50ms': total < 50,
        'bottleneck': max(latency, key=lambda x: x[1])[0]},
    'sensor_placement': placement,
    'joint_count': len(JOINTS),
    'sensor_count': 6,  # IMUs
    'camera_count': 2,  # Depth + RGB
    'wireless_protocol': 'ESP-NOW (2.4GHz, <250B packets, <10ms latency)',
    'processor': 'Raspberry Pi 5 + 6x ESP32-C3 + ESP32-S3 hub',
    'correction_model': 'Ridge regression (exported from correction_drone_kaggle.ipynb)',
}

with open('/kaggle/working/correction_drone_hardware.json', 'w') as f:
    json.dump(hardware_spec, f, indent=2)

print('✓ Hardware spec exported to /kaggle/working/correction_drone_hardware.json')
print(f'  BOM: ${drone_cost + suit_cost}')
print(f'  Drone life: {drone_life_min:.0f} min')
print(f'  Suit life: {suit_life_h:.1f} hrs')
print(f'  Latency: {total}ms ({"PASS" if total < 50 else "FAIL"})')
print(f'  Correction model: deployable on RPi5 at {8}ms inference')